# AAVAIL Capstone — EDA
Exploratory Data Analysis with visualizations.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from model.model import ingest_data, COUNTRIES

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Data

In [ ]:
df = ingest_data('all')
print(df.shape)
df.head()

## 2. Revenue Over Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['revenue', 'purchases', 'stream_views']):
    df.groupby(['year', 'month'])[col].sum().plot(ax=ax, marker='o')
    ax.set_title(f'{col} over time')
    ax.set_xlabel('(year, month)')
    ax.set_ylabel(col)
plt.tight_layout()
plt.savefig('../notebooks/eda_time_series.png', dpi=100)
plt.show()
print('Peak activity is in November-December (holiday season).')

## 3. Monthly Seasonality

In [ ]:
monthly = df.groupby('month')['revenue'].mean()
plt.figure()
monthly.plot(kind='bar', color='steelblue')
plt.title('Average Revenue by Month')
plt.xlabel('Month')
plt.ylabel('Avg Revenue')
plt.tight_layout()
plt.savefig('../notebooks/eda_monthly.png', dpi=100)
plt.show()

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[['revenue', 'purchases', 'stream_views', 'month', 'year']].corr(),
            annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../notebooks/eda_correlation.png', dpi=100)
plt.show()

## 5. Model Comparison — RMSE Visualization

In [ ]:
import json
from model.model import train_models
train_models()

with open('../model/metrics.json') as f:
    metrics = json.load(f)

# Plot RMSE comparison for 'all' country
country_m = metrics['all']
model_names = [k for k in country_m if k not in ('best_model', 'best_rmse')]
rmses = [country_m[m]['rmse'] for m in model_names]

plt.figure(figsize=(8, 4))
bars = plt.bar(model_names, rmses, color=['#2196F3', '#4CAF50', '#FF9800'])
best = country_m['best_model']
for bar, name in zip(bars, model_names):
    if name == best:
        bar.set_edgecolor('red')
        bar.set_linewidth(3)
plt.title('Model Comparison — RMSE (lower is better)\nRed border = best model')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.tight_layout()
plt.savefig('../notebooks/model_comparison.png', dpi=100)
plt.show()
print(f"Best model: {best} with RMSE={country_m['best_rmse']}")

## 6. Baseline vs Best Model

In [ ]:
from model.model import ingest_data, build_features, load_model
from sklearn.model_selection import train_test_split
import numpy as np

df = ingest_data('all')
X, y = build_features(df)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = load_model('all')
y_pred = model.predict(X_test)
baseline = np.full_like(y_test, y_test.mean())   # naive mean baseline

plt.figure(figsize=(12, 5))
idx = np.argsort(y_test)
plt.plot(y_test[idx], label='Actual', color='black', linewidth=1.5)
plt.plot(y_pred[idx], label='Best Model', color='steelblue', linewidth=1.5)
plt.plot(baseline[idx], label='Baseline (mean)', color='orange', linestyle='--')
plt.legend()
plt.title('Best Model vs Baseline Predictions')
plt.xlabel('Sample (sorted by actual)')
plt.ylabel('Revenue')
plt.tight_layout()
plt.savefig('../notebooks/baseline_vs_model.png', dpi=100)
plt.show()